# G3 · Inferencia física

**Spec:** [`docs/spec_G3_codex_physical_inference.md`](../docs/spec_G3_codex_physical_inference.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs12b_realigned`

Infiere L_acc y Ṁ (multilínea, límite = más restrictivo) y ajusta plantillas (diferido).

| | |
|---|---|
| **Entrada** | G2 + relaciones de acreción + modelos |
| **Salida (QC/productos)** | `stages/stage_g3_qc.json` |
| **Consume aguas abajo** | G4, G5 |


## Qué hace G3 y qué queda diferido

G3 es la **inferencia física**: de los flujos/límites de líneas (G2) infiere la **luminosidad de acreción L_acc** y la **tasa Ṁ** (relación Hα de Alcalá 2017), y *ajustaría* plantillas/atmósferas/tracks para SpT/Teff/masa — pero eso está **diferido (pendiente de librerías externas)**.

**Acreción en este objeto** (de su `stage_g3_qc.json`; `n/d` = G3 no ha calculado acreción para el objeto — su QC puede ser la variante *G3-real* de tipado espectral, con otro esquema):

| | |
|---|---|
| **L_acc** | 6.46e-07 L☉ (`detection`, de `n/d`, regla `n/d`) |
| **Ṁ p50** | 2.06e-13 M☉/yr (MC n=2000) |

**Diferencia con E3 (definicional):** G3 usa **5σ** del flujo de Hα de G2 **con** el factor de truncamiento de disco R_in=1.25; E3 usa Gumbel 99% **sin** R_in. Misma cadena física; el desfase entre 2.06e-13 (G3) y 5.84e-14 (E3) es de **definición**, no de física. Canónica **sin decidir** (usuario diferido, [`docs/2026-07-10_mdot_limit_definition_note.md`](../docs/2026-07-10_mdot_limit_definition_note.md)).

**Diferido → `not_constrained`:** atmósfera (BT-Settl), tracks (BHAC15/ATMO2020), plantillas (Luhman/Bonnefoy) — SpT/Teff/masa necesitan datos externos. **Por eso la clasificación de G4 es ambigua.** Provisional.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -c "from musepipe.stages.stage_g3_accretion import run_stage_g3_accretion; run_stage_g3_accretion('$RUN')"
```

Moderado (MC n=2000).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_g3_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -c "from musepipe.stages.stage_g3_accretion import run_stage_g3_accretion; run_stage_g3_accretion(\'$RUN\')"'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_g3_qc.json', RUN_ID)
nb.show(qc, keys=['mdot_p50_msun_yr', 'l_acc_lsun', 'combined_accretion', 'libraries', 'definition'], title='G3')


## Los términos de este QC, en físico

La cadena física es: **flujo de línea → luminosidad de línea → L_acc → Ṁ**. Cada flecha es una relación empírica de literatura con su dispersión, y ahí está casi todo el error.

| Término | Qué es | Por qué importa |
|---|---|---|
| `l_acc_lsun` | Luminosidad de acreción: la energía por segundo que libera el material al caer. | Se obtiene de la luminosidad de la línea con una relación calibrada en objetos donde ambas se midieron. |
| `mdot_p50_msun_yr` | La **mediana** (percentil 50) de la distribución de Ṁ del Monte Carlo, no un valor único. | Ṁ = L_acc·R/(G·M)·(1−R/R_in): cada ingrediente (masa, radio, extinción, distancia) entra con su incertidumbre, así que el resultado es una distribución. |
| `combined_accretion` | Cómo se combinan varias líneas en un solo número. | Con solo cotas superiores, la combinación es **la más restrictiva**, no un promedio. |
| `libraries` | Qué relaciones y modelos externos se usaron, con cita. | Cambiar de calibración cambia Ṁ en un factor: el número no significa nada sin decir con cuál se obtuvo. |
| `halpha_h03_consistency_v5` | Que G3 y E3 den lo mismo para Hα. | Miden lo mismo por caminos distintos; si divergen, hay un factor aplicado dos veces o ninguna. |
| `provisional` | Que el resultado depende de algo aún no cerrado. | Marca el número como no publicable todavía, aunque esté calculado. |


## Resultados que llevaron a la conclusión

Acreción, comparación con E3 y el estado de las librerías del `stage_g3_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G3', 'stages/stage_g3_qc.json'):
        q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
        ca = q['combined_accretion']
        print(f"acreción: {ca['kind']} L_acc = {ca['l_acc_lsun']:.2e} L☉ (de {ca['from_line']}, regla {ca['rule']})")
        print(f"Ṁ p50 = {q['mdot_p50_msun_yr']:.2e} M☉/yr (MC n={q['mc']['n']}); {q['n_lines_with_relation']} línea con relación")
        e3 = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
        e3_mdot = {L['method']: L['mdot'] for L in e3['limits']}[e3['canonical_method']]
        print(f"\ncomparación: E3 Ṁ={e3_mdot:.2e} (Gumbel99, sin R_in) vs G3 Ṁ={q['mdot_p50_msun_yr']:.2e} (5σ, con R_in) -> {q['mdot_p50_msun_yr']/e3_mdot:.2f}× definicional")
        print('\nlibrerías (tipado espectral):')
        for k, v in q['libraries'].items():
            print(f"   {k:20s} {v}")


## Plot 1 — E3 vs G3: la misma física, dos definiciones

Los dos límites de Ṁ de **este objeto**: E3 = 5.84e-14 (Gumbel 99%, sin R_in) y G3 = 2.06e-13 (5σ, con el factor R_in 1.25). El desfase es puramente **definicional** — misma cadena física. Canónica sin decidir.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
    e3 = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
    e3_mdot = {L['method']: L['mdot'] for L in e3['limits']}[e3['canonical_method']]
    g3_mdot = q.get('mdot_p50_msun_yr')
    if g3_mdot is None:
        raise KeyError('el QC de G3 de este objeto no trae acreción '
                       "('mdot_p50_msun_yr'): puede ser la variante G3-real de "
                       'tipado espectral. Sin par E3/G3 que comparar.')
    fig, ax = plt.subplots(figsize=(6.5, 4.3))
    bars = ax.bar(['E3\n(Gumbel 99%,\nsin R_in)', 'G3\n(5σ,\ncon R_in 1.25)'], [e3_mdot, g3_mdot],
                  color=['tab:blue', 'tab:green'])
    for b, v in zip(bars, [e3_mdot, g3_mdot]):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.02, f'{v:.2e}', ha='center', fontsize=10)
    ax.set_ylabel('Ṁ límite superior [M☉/yr]')
    ax.set_title(f'G3 · E3 vs G3: {g3_mdot/e3_mdot:.2f}× (definicional, misma física)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g3_accretion'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'e3_vs_g3.png', dpi=110); print('figura ->', outdir / 'e3_vs_g3.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — qué constriñe G3 y qué queda diferido

G3 **computa** la acreción (L_acc, Ṁ vía Alcalá) pero **difiere** el tipado espectral (atmósfera BT-Settl, tracks, plantillas) por falta de librerías externas → SpT/Teff/masa `not_constrained` → **G4 ambigua**.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g3_qc.json', RUN_ID)
    items = [('acreción (L_acc, Ṁ)', 'computado'),
             ('atmósfera (BT-Settl)', 'diferido'),
             ('tracks (BHAC15/ATMO2020)', 'diferido'),
             ('plantillas (Luhman/Bonnefoy)', 'diferido')]
    col = {'computado': 'tab:green', 'diferido': '0.7'}
    names = [i[0] for i in items]
    fig, ax = plt.subplots(figsize=(8, 3.2))
    ax.barh(names, [1] * len(names), color=[col[i[1]] for i in items])
    for i, (n, s) in enumerate(items):
        ax.text(0.5, i, f'{n}  →  {s}', ha='center', va='center', fontsize=9, color='w' if s == 'computado' else 'k', weight='bold')
    ax.set_xlim(0, 1); ax.set_xticks([]); ax.set_yticks([]); ax.invert_yaxis()
    ax.set_title('G3 · acreción computada; tipado espectral diferido (pending_libraries)')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g3_accretion'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'characterization_status.png', dpi=110); print('figura ->', outdir / 'characterization_status.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Ṁ p50 = 2.06e-13 M☉/yr** (5σ de G2 + factor R_in 1.25); difiere de E3 (5.84e-14) solo por DEFINICIÓN; canónica sin decidir. · [`docs/2026-07-10_mdot_limit_definition_note.md`](../docs/2026-07-10_mdot_limit_definition_note.md)
- L_acc detection = 6.46e-07 L☉ de `n/d` (regla `n/d`).
- Plantilla/atmósfera/tracks = `not_constrained` (pending_libraries: BT-Settl/BHAC15/Luhman-Bonnefoy diferidas) → G4 ambigua.


## Conclusión (registrada)

**G3 (este objeto): L_acc = 6.46e-07 L☉ (`detection`), Ṁ p50 = 2.06e-13 M☉/yr (5σ + R_in).** `n/d` = la acreción no está calculada en el QC de este objeto.

- **vs E3:** 5.84e-14 (Gumbel99, sin R_in) → la diferencia es de definición, no de física; canónica sin decidir.
- **Tipado espectral diferido:** atmósfera/tracks/plantillas `not_constrained` (falta de librerías externas).
- **Consecuencia:** sin SpT/Teff/masa espectroscópicos, la clasificación de G4 queda **ambigua** (planeta/BD/M no resuelto).
- **Downstream:** G4 (clasificación) y G5 (síntesis).
